# Ours — Natural Antibody–Antigen Datasets
## SAbDab · AbBind · SKEMPI · Benchmark — Comprehensive Analysis

All experiments with the **MutualTriStreamStrong** model (`mutual_strong.py`):
10-fold CV on SAbDab/AbBind/SKEMPI, full-train-on-SAbDab→test-on-benchmark, PLM comparison, regression-head comparison, architecture ablations (Two-stream, Concat+MLP), antigen shuffling, gating ablation, mutational impact, and the **All-CDR (H1+H2+H3)** pooling variant compared against mean-pool ESM-2.

In [ ]:
import os, sys, glob, pickle, warnings
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr, norm as _norm, ttest_ind
warnings.filterwarnings('ignore')
HERE = os.path.dirname(os.path.abspath('__file__'))
DEVICE = 'cuda'
print('Working dir:', HERE)

In [ ]:
# Publication style (Nature-like)
plt.rcParams.update({
    'font.family':'Arial','font.size':7,'axes.titlesize':8,'axes.labelsize':7,
    'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':6,'legend.frameon':False,
    'axes.linewidth':0.8,'axes.spines.top':False,'axes.spines.right':False,
    'figure.dpi':400,'savefig.dpi':400,'savefig.bbox':'tight','pdf.fonttype':42,'ps.fonttype':42})
PKD = r'pK$_d$'
C = {'ours':'#e74c3c','ours2':'#c0392b','two':'#2ecc71','concat':'#3498db',
     'rand':'#95a5a6','esm':'#2ecc71','progen':'#3498db','ablang':'#e67e22',
     'antiberty':'#f39c12','protbert':'#9b59b6','allcdr':'#e74c3c','meanpool':'#95a5a6'}
FIG = os.path.join(HERE,'figures_natural'); os.makedirs(FIG, exist_ok=True)
def save_fig(fig,name):
    for ext in ('pdf','png'): fig.savefig(os.path.join(FIG,f'{name}.{ext}'),bbox_inches='tight')
    print('saved',name)
def cv(path,metric='pearson'):
    d=pd.read_csv(path); r=d[d.metric==metric]['mean'].values; s=d[d.metric==metric]['std'].values
    return (float(r[0]) if len(r) else np.nan, float(s[0]) if len(s) else np.nan)
# Multi-seed results loader (mean +/- std over 5 seeds)
MS=os.path.join(HERE,'experiments','results')
_master=pd.read_csv(os.path.join(MS,'MASTER_SUMMARY.csv')) if os.path.isfile(os.path.join(MS,'MASTER_SUMMARY.csv')) else None
def ms_get(exp,ds,metric):
    if _master is None: return (np.nan,np.nan)
    r=_master[(_master.experiment==exp)&(_master.dataset==ds)]
    if not len(r): return (np.nan,np.nan)
    return float(r[f'{metric}_mean'].values[0]), float(r[f'{metric}_std'].values[0])
def panel(ax,letter):
    ax.text(0.02,0.97,letter,transform=ax.transAxes,fontsize=10,fontweight='bold',va='top',
            bbox=dict(boxstyle='round,pad=0.2',fc='white',alpha=0.7))
def fisher_p(r1,n1,r2,n2):
    z=(np.arctanh(np.clip(r1,-.999,.999))-np.arctanh(np.clip(r2,-.999,.999)))/np.sqrt(1/(n1-3)+1/(n2-3))
    return float(z), float(2*(1-_norm.cdf(abs(z))))
print('style ready ->', FIG)

---
## 1. Dataset Overview

In [ ]:
datasets = {'SAbDab':'pairs_sabdab.csv','AbBind':'pairs_abbind.csv',
            'SKEMPI':'pairs_skempi.csv','Benchmark':'pairs_benchmark.csv'}
info=[]
for name,fn in datasets.items():
    d=pd.read_csv(os.path.join(HERE,'datasets',fn))
    info.append({'Dataset':name,'n':len(d),'Y_min':d.Y.min(),'Y_max':d.Y.max(),'Y_mean':d.Y.mean()})
info=pd.DataFrame(info); print(info.to_string(index=False))

In [ ]:
fig,axes=plt.subplots(1,4,figsize=(9,2.2))
for ax,(name,fn),letter in zip(axes,datasets.items(),'ABCD'):
    d=pd.read_csv(os.path.join(HERE,'datasets',fn))
    ax.hist(d.Y,bins=25,color=C['ours'],alpha=0.8,edgecolor='white',lw=0.3)
    ax.axvline(d.Y.median(),color='k',ls='--',lw=1)
    ax.set_xlabel(f'Affinity ({PKD})'); ax.set_ylabel('Count' if letter=='A' else '')
    ax.set_title(f'{letter}  {name} (n={len(d)})',fontsize=8)
plt.tight_layout(); save_fig(fig,'natfig1_dataset_overview'); plt.show()

---
## 2. Main Model — 10-fold CV (SAbDab / AbBind / SKEMPI)
MutualTriStreamStrong (Ours), mean-pool ESM-2. **Multi-seed: mean ± std over 5 seeds.**

In [ ]:
main_cv={}
for ds in ['sabdab','abbind','skempi']:
    main_cv[ds]={m:ms_get('ours_meanpool_cv',ds,m) for m in ['pearson','spearman','rmse']}
for ds in main_cv:
    print(ds.upper(), {k:f'{v[0]:.4f}±{v[1]:.4f}' for k,v in main_cv[ds].items()})

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(7.2,2.6))
dss=['sabdab','abbind','skempi']; lbl=['SAbDab','AbBind','SKEMPI']
for ax,metric,ylab,letter in [(axes[0],'pearson','Pearson r','A'),
        (axes[1],'spearman','Spearman \u03c1','B'),(axes[2],'rmse',f'RMSE ({PKD})','C')]:
    vals=[main_cv[d][metric][0] for d in dss]; errs=[main_cv[d][metric][1] for d in dss]
    b=ax.bar(range(3),vals,yerr=errs,color=C['ours'],alpha=0.85,width=0.6,
             error_kw=dict(lw=1,capsize=3))
    ax.set_xticks(range(3)); ax.set_xticklabels(lbl); ax.set_ylabel(ylab)
    ax.set_title(f'{letter}  {ylab}',fontsize=8); panel(ax,letter)
    for i,v in enumerate(vals): ax.text(i,v+(errs[i] or 0)+0.01,f'{v:.3f}',ha='center',fontsize=6.5,fontweight='bold')
plt.tight_layout(w_pad=1.5); save_fig(fig,'natfig2_main_cv'); plt.show()

---
## 3. True vs Predicted (pooled 10-fold out-of-fold predictions)

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(7.2,2.6))
for ax,ds,letter in zip(axes,['sabdab','abbind','skempi'],'ABC'):
    p=os.path.join(HERE,f'results_mutual_strong/cv_{ds}/all_folds_predictions.csv')
    d=pd.read_csv(p); t=d['true_affinity'].values; pr=d['predicted_affinity'].values
    ax.scatter(t,pr,c=C['ours'],alpha=0.35,s=8,linewidths=0,rasterized=True)
    lims=[min(t.min(),pr.min())-0.3,max(t.max(),pr.max())+0.3]; ax.plot(lims,lims,'k--',lw=0.7,alpha=0.5)
    r,_=pearsonr(t,pr); rho,_=spearmanr(t,pr)
    ax.set_xlabel(f'Measured {PKD}'); ax.set_ylabel(f'Predicted {PKD}' if letter=='A' else '')
    ax.set_title(f'{letter}  {ds.upper()} (n={len(d)})',fontsize=8); panel(ax,letter)
    ax.text(0.05,0.88,f'r={r:.3f}\n\u03c1={rho:.3f}',transform=ax.transAxes,fontsize=6,va='top')
plt.tight_layout(w_pad=1.5); save_fig(fig,'natfig3_scatter'); plt.show()

---
## 4. PLM Comparison — ESM-2 / ProGen2 / AbLang / AntiBERTy / ProtBERT

In [ ]:
plm=pd.read_csv(os.path.join(HERE,'plm_comparison_results_mutual/cv_all_plms.csv'))
print(plm[['PLM','Dataset','CV_Pearson','CV_Spearman']].to_string(index=False))

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(7.2,2.8))
plms=['ESM-2','ProGen2','AbLang','AntiBERTy','ProtBERT']
pcol=[C['esm'],C['progen'],C['ablang'],C['antiberty'],C['protbert']]
for ax,ds,letter in zip(axes,['SABDAB','ABBIND','SKEMPI'],'ABC'):
    sub=plm[plm.Dataset==ds]
    vals=[float(sub[sub.PLM==p]['CV_Pearson'].values[0]) if p in sub.PLM.values else 0 for p in plms]
    errs=[float(sub[sub.PLM==p]['CV_Pearson_std'].values[0]) if p in sub.PLM.values else 0 for p in plms]
    ax.bar(range(len(plms)),vals,yerr=errs,color=pcol,alpha=0.85,width=0.7,error_kw=dict(lw=0.8,capsize=2))
    ax.set_xticks(range(len(plms))); ax.set_xticklabels(plms,rotation=30,ha='right',fontsize=6)
    ax.set_ylabel('Pearson r' if letter=='A' else ''); ax.set_title(f'{letter}  {ds}',fontsize=8); panel(ax,letter)
plt.tight_layout(w_pad=1.5); save_fig(fig,'natfig4_plm_comparison'); plt.show()

---
## 5. Architecture Comparison — Ours vs Two-stream vs Concat+MLP (with Fisher tests)

In [ ]:
arch={}
for ds in ['sabdab','abbind','skempi']:
    arch[ds]={
        'Ours':ms_get('ours_meanpool_cv',ds,'pearson'),  # multi-seed (5 seeds)
        'Two-stream':cv(os.path.join(HERE,f'results_twostream/cv_{ds}/cv_summary.csv')),
        'Concat+MLP':cv(os.path.join(HERE,f'results_baseline/cv_{ds}/cv_summary.csv'))}
nmap={'sabdab':578,'abbind':1089,'skempi':387}
for ds in arch:
    print(ds.upper(),{k:round(v[0],4) for k,v in arch[ds].items()})

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(7.2,2.8))
models=['Ours','Two-stream','Concat+MLP']; mcol=[C['ours'],C['two'],C['concat']]
for ax,ds,letter in zip(axes,['sabdab','abbind','skempi'],'ABC'):
    vals=[arch[ds][m][0] for m in models]; errs=[arch[ds][m][1] for m in models]
    ax.bar(range(3),vals,yerr=errs,color=mcol,alpha=0.85,width=0.6,error_kw=dict(lw=1,capsize=3))
    ax.set_xticks(range(3)); ax.set_xticklabels(models,rotation=20,ha='right',fontsize=6.5)
    ax.set_ylabel('Pearson r' if letter=='A' else ''); ax.set_title(f'{letter}  {ds.upper()}',fontsize=8); panel(ax,letter)
    # Fisher Ours vs Concat+MLP
    z,pv=fisher_p(arch[ds]['Ours'][0],nmap[ds],arch[ds]['Concat+MLP'][0],nmap[ds])
    if pv<0.05: ax.text(2,vals[2]+errs[2]+0.02,'*',ha='center',fontsize=11,fontweight='bold')
    for i,v in enumerate(vals): ax.text(i,v+errs[i]+0.005,f'{v:.3f}',ha='center',fontsize=5.5)
plt.tight_layout(w_pad=1.5); save_fig(fig,'natfig5_architecture'); plt.show()

---
## 6. Antigen Importance — Real vs Shuffled (Random) Antigen
Shuffling the antigen embedding quantifies reliance on antigen identity. **Multi-seed: mean ± std over 5 seeds.**

In [ ]:
ant={}
for ds in ['sabdab','abbind','skempi']:
    ant[ds]={'Real':ms_get('ours_meanpool_cv',ds,'pearson'),
             'Shuffled':ms_get('antigen_shuffle_cv',ds,'pearson')}
for ds in ant: print(ds.upper(),'real',round(ant[ds]['Real'][0],3),'shuffled',round(ant[ds]['Shuffled'][0],3))

In [ ]:
fig,ax=plt.subplots(figsize=(4.2,3.0))
dss=['sabdab','abbind','skempi']; x=np.arange(3); w=0.38
rv=[ant[d]['Real'][0] for d in dss]; re_=[ant[d]['Real'][1] for d in dss]
sv=[ant[d]['Shuffled'][0] for d in dss]; se=[ant[d]['Shuffled'][1] for d in dss]
ax.bar(x-w/2,rv,w,yerr=re_,color=C['ours'],alpha=0.85,label='Real antigen',error_kw=dict(lw=1,capsize=3))
ax.bar(x+w/2,sv,w,yerr=se,color=C['rand'],alpha=0.85,label='Shuffled antigen',error_kw=dict(lw=1,capsize=3))
for i in range(3):
    ax.text(i-w/2,rv[i]+re_[i]+0.01,f'{rv[i]:.2f}',ha='center',fontsize=6)
    ax.text(i+w/2,sv[i]+se[i]+0.01,f'{sv[i]:.2f}',ha='center',fontsize=6)
    d=rv[i]-sv[i]; ax.annotate(f'\u0394{d:+.2f}',(i,max(rv[i],sv[i])+0.06),ha='center',fontsize=6.5,
             color=C['ours'] if d>0.05 else 'gray',fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['SAbDab','AbBind','SKEMPI'])
ax.set_ylabel('Pearson r'); ax.set_title('Antigen importance: real vs shuffled',fontsize=8.5); ax.legend()
plt.tight_layout(); save_fig(fig,'natfig6_antigen_shuffle'); plt.show()

---
## 7. Generalization — Train on SAbDab, Test on Benchmark (38 complexes)
Key external-validation setting. **Ours: multi-seed (5 seeds), full-epoch protocol.**

In [ ]:
bench={'Ours':ms_get('ours_meanpool_benchmark','benchmark','pearson'),
       'Two-stream':cv(os.path.join(HERE,'results_twostream/benchmark/benchmark_multi_seed_summary.csv')),
       'Concat+MLP':cv(os.path.join(HERE,'results_baseline/benchmark/benchmark_summary.csv'))}
for k,v in bench.items(): print(k,'r=',round(v[0],4))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(7.0,3.0))
models=['Ours','Two-stream','Concat+MLP']; mcol=[C['ours'],C['two'],C['concat']]
ax=axes[0]
vals=[bench[m][0] for m in models]; errs=[bench[m][1] for m in models]
ax.bar(range(3),vals,yerr=errs,color=mcol,alpha=0.85,width=0.6,error_kw=dict(lw=1,capsize=3))
ax.axhline(0.60,color='k',ls=':',lw=1,label='~60% reference')
ax.set_xticks(range(3)); ax.set_xticklabels(models,rotation=20,ha='right',fontsize=6.5)
ax.set_ylabel('Benchmark Pearson r'); ax.set_title('A  SAbDab\u2192Benchmark transfer',fontsize=8)
ax.legend(fontsize=6); panel(ax,'A')
for i,v in enumerate(vals): ax.text(i,v+errs[i]+0.01,f'{v:.3f}',ha='center',fontsize=6.5,fontweight='bold')
# PLM on benchmark
ax=axes[1]
pb=pd.read_csv(os.path.join(HERE,'plm_comparison_results_mutual/benchmark_all_plms.csv'))
ax.bar(range(len(pb)),pb['Bench_Pearson'],yerr=pb['Bench_Pearson_std'],
       color=[C['esm'],C['progen'],C['ablang'],C['antiberty'],C['protbert']][:len(pb)],
       alpha=0.85,width=0.65,error_kw=dict(lw=0.8,capsize=2))
ax.set_xticks(range(len(pb))); ax.set_xticklabels(pb['PLM'],rotation=30,ha='right',fontsize=6)
ax.set_ylabel('Benchmark Pearson r'); ax.set_title('B  PLM on benchmark',fontsize=8); panel(ax,'B')
plt.tight_layout(w_pad=2); save_fig(fig,'natfig7_benchmark'); plt.show()

---
## 8. Regression-Head Comparison (cosine vs MLP / RF / LightGBM / XGBoost)
Heads trained on the frozen tri-stream representation (SAaIntDB reference results).

In [ ]:
heads={'Cosine (Ours)':cv(os.path.join(HERE,'results_mutual_strong_saaintdb/cv_summary.csv')),
       'MLP':cv(os.path.join(HERE,'results_mutual_strong_saaintdb_head/mlp/cv_summary.csv')),
       'Random Forest':cv(os.path.join(HERE,'results_mutual_strong_saaintdb_head/random_forest/cv_summary.csv')),
       'LightGBM':cv(os.path.join(HERE,'results_mutual_strong_saaintdb_head/lightgbm/cv_summary.csv')),
       'XGBoost':cv(os.path.join(HERE,'results_mutual_strong_saaintdb_head/xgboost/cv_summary.csv'))}
for k,v in heads.items(): print(k,'r=',round(v[0],4))

In [ ]:
fig,ax=plt.subplots(figsize=(4.6,3.0))
names=list(heads); vals=[heads[k][0] for k in names]; errs=[heads[k][1] for k in names]
cols=[C['ours']]+[C['concat']]*4
ax.bar(range(len(names)),vals,yerr=errs,color=cols,alpha=0.85,width=0.65,error_kw=dict(lw=1,capsize=3))
ax.set_xticks(range(len(names))); ax.set_xticklabels(names,rotation=30,ha='right',fontsize=6.5)
ax.set_ylabel('Pearson r'); ax.set_title('Regression-head comparison (SAaIntDB)',fontsize=8.5)
for i,v in enumerate(vals): ax.text(i,v+errs[i]+0.005,f'{v:.3f}',ha='center',fontsize=6)
plt.tight_layout(); save_fig(fig,'natfig8_heads'); plt.show()

---
## 9. Mutational Impact — Per-Complex Error Analysis (SKEMPI & AbBind)
Out-of-fold residuals reveal which complexes/mutations the model captures well.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(7.2,3.0))
for ax,ds,letter in zip(axes,['skempi','abbind'],'AB'):
    d=pd.read_csv(os.path.join(HERE,f'results_mutual_strong/cv_{ds}/all_folds_predictions.csv'))
    d['abs_err']=d['error'].abs()
    ax.hist(d['error'],bins=40,color=C['ours'],alpha=0.8,edgecolor='white',lw=0.3)
    ax.axvline(0,color='k',ls='--',lw=0.8)
    ax.set_xlabel(f'Residual (true \u2212 pred) {PKD}'); ax.set_ylabel('Count' if letter=='A' else '')
    ax.set_title(f'{letter}  {ds.upper()} residuals (MAE={d.abs_err.mean():.2f})',fontsize=8); panel(ax,letter)
plt.tight_layout(w_pad=1.5); save_fig(fig,'natfig9_mutational_residuals'); plt.show()

---
## 10. Gating Mechanism — Learned vs Fixed/Open/Closed Gate
Gate-perturbation ablation (reference results on SAaIntDB; same architecture).

In [ ]:
gp=os.path.join(HERE,'figures_out_saaintdb/additional_experiments/gate_perturbation_errorbars.csv')
have_gate=os.path.isfile(gp)
if have_gate:
    g=pd.read_csv(gp)
    print(g[g.metric=='pearson'][['split','gate_mode','mean','ci95']].to_string(index=False))
else: print('gate perturbation CSV not found')

In [ ]:
if have_gate:
    modes=['learned','fixed_0.5','open_1.0','closed_0.0','random_uniform']
    labels=['Learned\n(Ours)','Fixed 0.5','Open 1.0','Closed 0.0','Random']
    gcol=[C['ours'],C['concat'],C['two'],C['rand'],C['antiberty']]
    fig,axes=plt.subplots(1,2,figsize=(7.2,3.0))
    for ax,split,letter in [(axes[0],'Random','A'),(axes[1],'Cold','B')]:
        vals=[];cis=[]
        for m in modes:
            row=g[(g.split==split)&(g.gate_mode==m)&(g.metric=='pearson')]
            vals.append(float(row['mean'].values[0]) if len(row) else np.nan)
            cis.append(float(row['ci95'].values[0]) if len(row) else 0)
        b=ax.bar(range(len(modes)),vals,yerr=cis,color=gcol,alpha=0.85,width=0.65,error_kw=dict(lw=1,capsize=3))
        b[0].set_edgecolor('black'); b[0].set_linewidth(1.5)
        ax.set_xticks(range(len(modes))); ax.set_xticklabels(labels,fontsize=6)
        ax.set_ylabel('Pearson r' if letter=='A' else ''); ax.set_title(f'{letter}  {split} split',fontsize=8); panel(ax,letter)
    plt.tight_layout(w_pad=1.5); save_fig(fig,'natfig10_gating'); plt.show()

---
## 11. All-CDR (H1+H2+H3) vs Mean-Pool — SAbDab / AbBind / SKEMPI / Benchmark
CDR-aware heavy pooling (ANARCI IMGT). **Multi-seed: mean ± std over 5 seeds, error bars.**

In [ ]:
# Map each dataset to its (meanpool_exp, allcdr_exp) multi-seed experiments
pairs={'sabdab':('ours_meanpool_cv','ours_allcdr_cv'),
       'abbind':('ours_meanpool_cv','ours_allcdr_cv'),
       'skempi':('ours_meanpool_cv','ours_allcdr_cv'),
       'benchmark':('ours_meanpool_benchmark','ours_allcdr_benchmark')}
rows=[]
for ds,(mp_e,cd_e) in pairs.items():
    mp=ms_get(mp_e,ds,'pearson'); cd=ms_get(cd_e,ds,'pearson')
    rows.append({'dataset':ds,'meanpool':mp[0],'meanpool_std':mp[1],
                 'allcdr':cd[0],'allcdr_std':cd[1],'delta':cd[0]-mp[0]})
allcdr_ms=pd.DataFrame(rows); print(allcdr_ms.to_string(index=False))

In [ ]:
order=['sabdab','abbind','skempi','benchmark']; lbl=['SAbDab','AbBind','SKEMPI','Benchmark']
fig,axes=plt.subplots(1,2,figsize=(7.6,3.0))
x=np.arange(len(order)); w=0.38
base =[float(allcdr_ms[allcdr_ms.dataset==d]['meanpool'].values[0]) for d in order]
bstd =[float(allcdr_ms[allcdr_ms.dataset==d]['meanpool_std'].values[0]) for d in order]
cdr  =[float(allcdr_ms[allcdr_ms.dataset==d]['allcdr'].values[0]) for d in order]
cstd =[float(allcdr_ms[allcdr_ms.dataset==d]['allcdr_std'].values[0]) for d in order]
ax=axes[0]
ax.bar(x-w/2,base,w,yerr=bstd,color=C['meanpool'],alpha=0.85,label='Mean-pool',error_kw=dict(lw=1,capsize=3))
ax.bar(x+w/2,cdr,w,yerr=cstd,color=C['allcdr'],alpha=0.85,label='All-CDR',error_kw=dict(lw=1,capsize=3))
ax.axhline(0.60,color='k',ls=':',lw=1,alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(lbl,fontsize=6.5); ax.set_ylabel('Pearson r')
ax.set_title('A  All-CDR vs mean-pool (mean ± std)',fontsize=8); ax.legend(fontsize=6); panel(ax,'A')
for i in range(len(order)):
    ax.text(x[i]-w/2,base[i]+bstd[i]+0.01,f'{base[i]:.2f}',ha='center',fontsize=5.3)
    ax.text(x[i]+w/2,cdr[i]+cstd[i]+0.01,f'{cdr[i]:.2f}',ha='center',fontsize=5.3)
# Panel B: delta with propagated error
ax=axes[1]; dl=[c-b for c,b in zip(cdr,base)]
derr=[np.sqrt(a**2+bb**2) for a,bb in zip(cstd,bstd)]
cols=[C['allcdr'] if d>0 else C['concat'] for d in dl]
ax.barh(range(len(order)),dl,xerr=derr,color=cols,alpha=0.85,height=0.6,error_kw=dict(lw=1,capsize=3))
ax.set_yticks(range(len(order))); ax.set_yticklabels(lbl,fontsize=6.5); ax.axvline(0,color='k',lw=0.6)
ax.set_xlabel('\u0394 Pearson r (All-CDR \u2212 mean-pool)'); ax.set_title('B  CDR-aware gain',fontsize=8); panel(ax,'B')
for i,d in enumerate(dl):
    ax.text(d+(0.004 if d>=0 else -0.004),i,f'{d:+.3f}',va='center',
             ha='left' if d>=0 else 'right',fontsize=6.5,fontweight='bold')
plt.tight_layout(w_pad=2); save_fig(fig,'natfig11_allcdr_vs_meanpool'); plt.show()

---
## 12. All-CDR Benchmark Scatter — SAbDab→Benchmark

In [ ]:
bp=os.path.join(HERE,'results_natural_allcdr/benchmark_allcdr_preds.csv')
if os.path.isfile(bp):
    d=pd.read_csv(bp); t=d['true'].values; pr=d['pred'].values
    r,_=pearsonr(t,pr); rho,_=spearmanr(t,pr)
    fig,ax=plt.subplots(figsize=(3.6,3.4))
    ax.scatter(t,pr,c=C['allcdr'],alpha=0.7,s=30,edgecolors='white',lw=0.5)
    lims=[min(t.min(),pr.min())-0.3,max(t.max(),pr.max())+0.3]; ax.plot(lims,lims,'k--',lw=0.7,alpha=0.5)
    m,b=np.polyfit(t,pr,1); xs=np.linspace(*lims,100); ax.plot(xs,m*xs+b,color='k',lw=1)
    ax.set_xlabel(f'Measured {PKD}'); ax.set_ylabel(f'Predicted {PKD}')
    ax.set_title(f'All-CDR: SAbDab\u2192Benchmark\nr={r:.3f}  \u03c1={rho:.3f}  n={len(d)}',fontsize=8)
    plt.tight_layout(); save_fig(fig,'natfig12_allcdr_benchmark_scatter'); plt.show()
else: print('All-CDR benchmark preds not ready.')

---
## 13. Summary of All Experiments

In [ ]:
rows=[]
for ds in ['sabdab','abbind','skempi']:
    p=main_cv[ds]['pearson']; s=main_cv[ds]['spearman']
    rows.append({'Experiment':f'Ours 10-fold CV ({ds})',
                 'Pearson r':f'{p[0]:.3f}±{p[1]:.3f}','Spearman':f'{s[0]:.3f}±{s[1]:.3f}'})
bo=bench['Ours']
rows.append({'Experiment':'Ours SAbDab\u2192Benchmark','Pearson r':f'{bo[0]:.3f}±{bo[1]:.3f}','Spearman':'-'})
for ds in ['sabdab','abbind','skempi']:
    rows.append({'Experiment':f'Concat+MLP ({ds})','Pearson r':f'{arch[ds]["Concat+MLP"][0]:.3f}','Spearman':'-'})
for _,r in allcdr_ms.iterrows():
    rows.append({'Experiment':f'All-CDR ({r.dataset})',
                 'Pearson r':f'{r.allcdr:.3f}±{r.allcdr_std:.3f}','Spearman':f'\u0394={r.delta:+.3f} vs mean-pool'})
print(pd.DataFrame(rows).to_string(index=False))